In [150]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
1
NVIDIA GeForce RTX 4060 Laptop GPU


In [151]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for training in this notebook.")

device = torch.device("cuda")
print(device)
print(torch.cuda.get_device_name(0))

cuda
NVIDIA GeForce RTX 4060 Laptop GPU


In [152]:
import pandas as pd
import numpy as np

ratings = pd.read_csv(
    "../data/processed/ratings_master.csv"
)

ratings.head()

,User-ID,ISBN,Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [153]:
explicit_ratings = ratings[
    ratings["Rating"] > 0
].copy()

print(explicit_ratings.shape)

(383842, 3)


In [154]:
user_counts = (
    explicit_ratings
    .groupby("User-ID")
    .size()
)

In [155]:
user_counts.describe()

count    68091.000000
mean         5.637191
std         41.742511
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max       6943.000000
dtype: float64

In [156]:
thresholds = [5, 10, 20]

for t in thresholds:

    active_users = user_counts[
        user_counts >= t
    ].index

    subset = explicit_ratings[
        explicit_ratings["User-ID"]
        .isin(active_users)
    ]

    print(f"\nThreshold >= {t}")

    print("Users:",
          subset["User-ID"].nunique())

    print("Books:",
          subset["ISBN"].nunique())

    print("Ratings:",
          len(subset))


Threshold >= 5
Users: 12787
Books: 131372
Ratings: 302218

Threshold >= 10
Users: 6589
Books: 121075
Ratings: 261899

Threshold >= 20
Users: 3305
Books: 108380
Ratings: 217729


In [157]:
datasets = {
    "All": explicit_ratings,
    ">=5": explicit_ratings[
        explicit_ratings["User-ID"].isin(
            user_counts[user_counts >= 5].index
        )
    ],
    ">=10": explicit_ratings[
        explicit_ratings["User-ID"].isin(
            user_counts[user_counts >= 10].index
        )
    ],
    ">=20": explicit_ratings[
        explicit_ratings["User-ID"].isin(
            user_counts[user_counts >= 20].index
        )
    ]
}

for name, df in datasets.items():

    users = df["User-ID"].nunique()
    books = df["ISBN"].nunique()
    ratings = len(df)

    density = (
        ratings / (users * books)
    ) * 100

    print(
        f"{name}: "
        f"{density:.4f}%"
    )

All: 0.0038%
>=5: 0.0180%
>=10: 0.0328%
>=20: 0.0608%


In [158]:
active_users = user_counts[
    user_counts >= 10
].index

ratings_cf = explicit_ratings[
    explicit_ratings["User-ID"].isin(active_users)
].copy()

print(ratings_cf.shape)

(261899, 3)


In [159]:
user2idx = {
    user: idx
    for idx, user in enumerate(
        ratings_cf["User-ID"].unique()
    )
}

idx2user = {
    idx: user
    for user, idx in user2idx.items()
}

In [160]:
book2idx = {
    isbn: idx
    for idx, isbn in enumerate(
        ratings_cf["ISBN"].unique()
    )
}

idx2book = {
    idx: isbn
    for isbn, idx in book2idx.items()
}

In [161]:
ratings_cf["user_idx"] = (
    ratings_cf["User-ID"]
    .map(user2idx)
)

ratings_cf["book_idx"] = (
    ratings_cf["ISBN"]
    .map(book2idx)
)


In [162]:
ratings_cf.head()

,User-ID,ISBN,Rating,user_idx,book_idx
99,276822,0060096195,10,0,0
100,276822,0141310340,9,0,1
101,276822,0142302198,10,0,2
102,276822,0156006065,9,0,3
103,276822,0375821813,9,0,4


In [163]:
ratings_cf.shape

ratings_cf.head()

,User-ID,ISBN,Rating,user_idx,book_idx
99,276822,0060096195,10,0,0
100,276822,0141310340,9,0,1
101,276822,0142302198,10,0,2
102,276822,0156006065,9,0,3
103,276822,0375821813,9,0,4


In [164]:
from sklearn.model_selection import train_test_split

train_rows = []
test_rows = []

for user_id, group in ratings_cf.groupby("user_idx"):

    if len(group) < 2:
        continue

    test_sample = group.sample(
        n=1,
        random_state=42
    )

    train_sample = group.drop(
        test_sample.index
    )

    train_rows.append(train_sample)
    test_rows.append(test_sample)

train_df = pd.concat(train_rows)
test_df = pd.concat(test_rows)

In [165]:
print("Train:", train_df.shape)
print("Test:", test_df.shape)

print(
    "Users in Train:",
    train_df["user_idx"].nunique()
)

print(
    "Users in Test:",
    test_df["user_idx"].nunique()
)

Train: (255310, 5)
Test: (6589, 5)
Users in Train: 6589
Users in Test: 6589


In [166]:
num_users = ratings_cf["user_idx"].nunique()
num_books = ratings_cf["book_idx"].nunique()

In [167]:
train_df.shape
test_df.shape

train_df["user_idx"].nunique()
test_df["user_idx"].nunique()

6589

In [168]:
print("Users:", ratings_cf["user_idx"].nunique())
print("Books:", ratings_cf["book_idx"].nunique())

Users: 6589
Books: 121075


In [169]:
import torch

train_users = torch.tensor(
    train_df["user_idx"].values,
    dtype=torch.long
)

train_books = torch.tensor(
    train_df["book_idx"].values,
    dtype=torch.long
)

train_ratings = torch.tensor(
    train_df["Rating"].values,
    dtype=torch.float32
)

In [170]:
test_users = torch.tensor(
    test_df["user_idx"].values,
    dtype=torch.long
)

test_books = torch.tensor(
    test_df["book_idx"].values,
    dtype=torch.long
)

test_ratings = torch.tensor(
    test_df["Rating"].values,
    dtype=torch.float32
)

In [171]:
import torch
import torch.nn as nn


class MatrixFactorization(nn.Module):

    def __init__(
        self,
        num_users,
        num_books,
        embedding_dim=50
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.book_embedding = nn.Embedding(
            num_books,
            embedding_dim
        )

        self.user_bias = nn.Embedding(
            num_users,
            1
        )

        self.book_bias = nn.Embedding(
            num_books,
            1
        )

    def forward(
        self,
        users,
        books
    ):

        user_vecs = self.user_embedding(users)

        book_vecs = self.book_embedding(books)

        interaction = (
            user_vecs * book_vecs
        ).sum(dim=1)

        user_bias = (
            self.user_bias(users)
            .squeeze()
        )

        book_bias = (
            self.book_bias(books)
            .squeeze()
        )

        return (
            interaction
            + user_bias
            + book_bias
        )

In [172]:
model = MatrixFactorization(
    num_users=6589,
    num_books=121075,
    embedding_dim=50
)

In [173]:
criterion = nn.MSELoss()

In [174]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [175]:
epochs = 10

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    predictions = model(
        train_users,
        train_books
    )

    loss = criterion(
        predictions,
        train_ratings
    )

    loss.backward()

    optimizer.step()

    print(
        f"Epoch {epoch+1}/{epochs}",
        f"Loss: {loss.item():.4f}"
    )

Epoch 1/10 Loss: 116.0905
Epoch 2/10 Loss: 115.4507
Epoch 3/10 Loss: 114.8140
Epoch 4/10 Loss: 114.1804
Epoch 5/10 Loss: 113.5501
Epoch 6/10 Loss: 112.9231
Epoch 7/10 Loss: 112.2993
Epoch 8/10 Loss: 111.6788
Epoch 9/10 Loss: 111.0617
Epoch 10/10 Loss: 110.4480


In [176]:
model.eval()

with torch.no_grad():

    preds = model(
        test_users,
        test_books
    )

    rmse = torch.sqrt(
        ((preds - test_ratings) ** 2).mean()
    )

print("RMSE:", rmse.item())

RMSE: 10.565898895263672


In [177]:
train_ratings.mean()

tensor(7.7198)

In [178]:
with torch.no_grad():

    preds = model(
        test_users,
        test_books
    )

print(preds[:20])

tensor([ -0.4718,  -5.5962,  11.5852,  -1.1676,  -2.5658,  -8.1209,  -0.4334,
         -0.5221, -10.0938,  -6.2819,  -7.4639,   1.2884,  13.3382,  -3.3089,
         16.8714,   8.7985,  13.9180,  -9.9252,   6.2025,  -2.5813])


In [179]:
print(test_ratings[:20])

tensor([ 8.,  8.,  5.,  5.,  6., 10.,  6., 10.,  6.,  6., 10.,  8.,  5.,  9.,
         7., 10.,  7.,  7.,  8., 10.])


In [180]:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

train_dataset = TensorDataset(
    train_users,
    train_books,
    train_ratings
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2048,
    shuffle=True,
    pin_memory=True,
    num_workers=2,
    persistent_workers=True
)

In [181]:
global_mean = train_ratings.mean().item()

In [182]:
class MatrixFactorization(nn.Module):

    def __init__(
        self,
        num_users,
        num_books,
        global_mean,
        embedding_dim=50
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.book_embedding = nn.Embedding(
            num_books,
            embedding_dim
        )

        self.user_bias = nn.Embedding(
            num_users,
            1
        )

        self.book_bias = nn.Embedding(
            num_books,
            1
        )

        self.global_mean = global_mean

    def forward(self, users, books):

        user_vecs = self.user_embedding(users)

        book_vecs = self.book_embedding(books)

        interaction = (
            user_vecs * book_vecs
        ).sum(dim=1)

        user_bias = (
            self.user_bias(users)
            .squeeze()
        )

        book_bias = (
            self.book_bias(books)
            .squeeze()
        )

        pred = (
            self.global_mean
            + interaction
            + user_bias
            + book_bias
        )

        return pred

In [183]:
model = MatrixFactorization(
    num_users=6589,
    num_books=121075,
    global_mean=global_mean,
    embedding_dim=50
).to(device)

In [184]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

In [185]:
epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for users, books, ratings in train_loader:

        users = users.to(device, non_blocking=True)
        books = books.to(device, non_blocking=True)
        ratings = ratings.to(device, non_blocking=True)

        optimizer.zero_grad()

        preds = model(
            users,
            books
        )

        loss = criterion(
            preds,
            ratings
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}",
        f"Loss: {total_loss:.4f}"
    )

Epoch 1 Loss: 6485.0164
Epoch 2 Loss: 5816.0114
Epoch 3 Loss: 5192.0729
Epoch 4 Loss: 4635.9368
Epoch 5 Loss: 4143.8086
Epoch 6 Loss: 3708.3752
Epoch 7 Loss: 3323.4347
Epoch 8 Loss: 2981.1364
Epoch 9 Loss: 2676.7674
Epoch 10 Loss: 2405.0653


In [186]:
model.eval()

with torch.no_grad():

    preds = model(
        test_users.to(device, non_blocking=True),
        test_books.to(device, non_blocking=True)
    )

    rmse = torch.sqrt(
        ((preds - test_ratings.to(device, non_blocking=True)) ** 2).mean()
    )

print("RMSE:", rmse.item())

RMSE: 5.64402961730957


In [187]:
print(preds[:20])
print(test_ratings[:20])

tensor([ 8.4983,  5.9569,  4.6133,  5.8043,  7.8126, 21.6786,  6.4660,  0.4333,
         7.5292,  7.4793, 19.0493, 10.8630,  7.9717,  6.3192, 11.9582, 14.8427,
        10.5563, 13.9814,  2.3134,  5.2016], device='cuda:0')
tensor([ 8.,  8.,  5.,  5.,  6., 10.,  6., 10.,  6.,  6., 10.,  8.,  5.,  9.,
         7., 10.,  7.,  7.,  8., 10.])


In [188]:
preds_clipped = torch.clamp(
    preds,
    min=1,
    max=10
)

rmse_clipped = torch.sqrt(
    ((preds_clipped - test_ratings.to(device, non_blocking=True)) ** 2).mean()
)

print("Clipped RMSE:",
      rmse_clipped.item())

Clipped RMSE: 3.6593432426452637


In [189]:
book_counts = (
    ratings_cf
    .groupby("ISBN")
    .size()
)

book_counts.describe()

count    121075.000000
mean          2.163114
std           4.843571
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         330.000000
dtype: float64

In [190]:
active_books = book_counts[
    book_counts >= 5
].index

In [191]:
ratings_dense = ratings_cf[
    ratings_cf["ISBN"]
    .isin(active_books)
].copy()

In [192]:
print("Users:",
      ratings_dense["User-ID"].nunique())

print("Books:",
      ratings_dense["ISBN"].nunique())

print("Ratings:",
      len(ratings_dense))

Users: 6436
Books: 9009
Ratings: 108344


In [193]:
book_counts.describe()


count    121075.000000
mean          2.163114
std           4.843571
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         330.000000
dtype: float64

In [194]:
active_books = book_counts[
    book_counts >= 5
].index

ratings_dense = ratings_cf[
    ratings_cf["ISBN"].isin(active_books)
]

print(ratings_dense["User-ID"].nunique())
print(ratings_dense["ISBN"].nunique())
print(len(ratings_dense))

6436
9009
108344


In [195]:
dense_user_counts = (
    ratings_dense
    .groupby("User-ID")
    .size()
)

dense_book_counts = (
    ratings_dense
    .groupby("ISBN")
    .size()
)

print("Users")
print(dense_user_counts.describe())

print("\nBooks")
print(dense_book_counts.describe())

Users
count    6436.000000
mean       16.834058
std        39.303941
min         1.000000
25%         5.000000
50%         9.000000
75%        18.000000
max      2436.000000
dtype: float64

Books
count    9009.000000
mean       12.026196
std        14.263046
min         5.000000
25%         6.000000
50%         8.000000
75%        13.000000
max       330.000000
dtype: float64


In [196]:
user2idx = {
    user: idx
    for idx, user in enumerate(
        ratings_dense["User-ID"].unique()
    )
}

In [197]:
book2idx = {
    isbn: idx
    for idx, isbn in enumerate(
        ratings_dense["ISBN"].unique()
    )
}

In [198]:
ratings_dense["user_idx"] = (
    ratings_dense["User-ID"]
    .map(user2idx)
)

ratings_dense["book_idx"] = (
    ratings_dense["ISBN"]
    .map(book2idx)
)

In [199]:
train_rows = []
test_rows = []

for user_id, group in ratings_dense.groupby("user_idx"):

    if len(group) < 2:
        continue

    test_sample = group.sample(
        n=1,
        random_state=42
    )

    train_sample = group.drop(
        test_sample.index
    )

    train_rows.append(train_sample)
    test_rows.append(test_sample)

train_df = pd.concat(train_rows)
test_df = pd.concat(test_rows)

In [200]:
print(train_df.shape)
print(test_df.shape)

print(
    train_df["user_idx"].nunique()
)

print(
    test_df["user_idx"].nunique()
)

(101908, 5)
(6197, 5)
6197
6197


In [201]:
num_users = ratings_dense["user_idx"].nunique()
num_books = ratings_dense["book_idx"].nunique()

density = (
    len(ratings_dense)
    /
    (num_users * num_books)
) * 100

print("Users:", num_users)
print("Books:", num_books)
print("Density:", density)

Users: 6436
Books: 9009
Density: 0.18685823533555043


In [202]:
train_users = torch.tensor(
    train_df["user_idx"].values,
    dtype=torch.long
)

train_books = torch.tensor(
    train_df["book_idx"].values,
    dtype=torch.long
)

train_ratings = torch.tensor(
    train_df["Rating"].values,
    dtype=torch.float32
)

In [203]:
test_users = torch.tensor(
    test_df["user_idx"].values,
    dtype=torch.long
)

test_books = torch.tensor(
    test_df["book_idx"].values,
    dtype=torch.long
)

test_ratings = torch.tensor(
    test_df["Rating"].values,
    dtype=torch.float32
)

In [204]:
from torch.utils.data import (
    TensorDataset,
    DataLoader
)

train_dataset = TensorDataset(
    train_users,
    train_books,
    train_ratings
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2048,
    shuffle=True,
    pin_memory=True,
    num_workers=2,
    persistent_workers=True
)

In [205]:
global_mean = (
    train_ratings.mean()
    .item()
)

print(global_mean)

7.838815212249756


In [206]:
model = MatrixFactorization(
    num_users=ratings_dense["user_idx"].nunique(),
    num_books=ratings_dense["book_idx"].nunique(),
    global_mean=global_mean,
    embedding_dim=50
).to(device)

In [207]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

In [208]:
epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for users, books, ratings in train_loader:

        users = users.to(device, non_blocking=True)
        books = books.to(device, non_blocking=True)
        ratings = ratings.to(device, non_blocking=True)

        optimizer.zero_grad()

        preds = model(users, books)

        loss = criterion(preds, ratings)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Avg Loss: {avg_loss:.4f}"
    )

Epoch 1/10 | Avg Loss: 54.0870
Epoch 2/10 | Avg Loss: 50.9235
Epoch 3/10 | Avg Loss: 48.0715
Epoch 4/10 | Avg Loss: 45.3656
Epoch 5/10 | Avg Loss: 42.8104
Epoch 6/10 | Avg Loss: 40.3882
Epoch 7/10 | Avg Loss: 38.1458
Epoch 8/10 | Avg Loss: 36.0058
Epoch 9/10 | Avg Loss: 34.0149
Epoch 10/10 | Avg Loss: 32.1220


In [209]:
model.eval()

with torch.no_grad():

    preds = model(
        test_users.to(device, non_blocking=True),
        test_books.to(device, non_blocking=True)
    )

    rmse = torch.sqrt(
        ((preds - test_ratings.to(device, non_blocking=True)) ** 2).mean()
    )

print("RMSE:", rmse.item())

RMSE: 6.922667503356934


In [210]:
preds_clipped = torch.clamp(
    preds,
    min=1,
    max=10
)

rmse_clipped = torch.sqrt(
    ((preds_clipped - test_ratings.to(device, non_blocking=True)) ** 2).mean()
)

print("Clipped RMSE:",
      rmse_clipped.item())

Clipped RMSE: 4.080382347106934


In [211]:
print(preds[:20])
print(test_ratings[:20])

tensor([ 9.3094,  8.2542,  9.4279, 13.1865, 15.1269,  3.5912,  7.6853,  3.9785,
         7.7882, 12.6444, 11.4605,  1.3092, 10.0526, 11.6510,  8.4101, 15.6452,
         5.6458, 10.6988, -3.7199,  4.8797], device='cuda:0')
tensor([10.,  8.,  6.,  2.,  9.,  8.,  6.,  5., 10.,  8.,  4.,  8.,  7., 10.,
         7.,  7.,  4.,  5., 10.,  7.])


In [212]:
with torch.no_grad():

    user_norms = (
        model.user_embedding.weight.norm(dim=1)
    )

    book_norms = (
        model.book_embedding.weight.norm(dim=1)
    )

print(user_norms.mean())
print(book_norms.mean())

tensor(6.7206, device='cuda:0')
tensor(6.8384, device='cuda:0')


In [213]:
global_mean = train_ratings.mean()

baseline_rmse = torch.sqrt(
    (
        (test_ratings - global_mean) ** 2
    ).mean()
)

print(baseline_rmse)

tensor(1.8307)


In [214]:
import torch
import torch.nn as nn

class BiasModel(nn.Module):

    def __init__(self, num_users, num_books, global_mean):
        super().__init__()

        self.user_bias = nn.Embedding(
            num_users,
            1
        )

        self.book_bias = nn.Embedding(
            num_books,
            1
        )

        self.global_mean = global_mean

    def forward(self, users, books):

        user_bias = (
            self.user_bias(users)
            .squeeze()
        )

        book_bias = (
            self.book_bias(books)
            .squeeze()
        )

        return (
            self.global_mean
            + user_bias
            + book_bias
        )

In [215]:
global_mean = train_ratings.mean().item()

bias_model = BiasModel(
    num_users=ratings_dense["user_idx"].nunique(),
    num_books=ratings_dense["book_idx"].nunique(),
    global_mean=global_mean
).to(device)

In [216]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    bias_model.parameters(),
    lr=0.001
)

In [217]:
epochs = 10

for epoch in range(epochs):

    bias_model.train()

    total_loss = 0

    for users, books, ratings in train_loader:

        users = users.to(device, non_blocking=True)
        books = books.to(device, non_blocking=True)
        ratings = ratings.to(device, non_blocking=True)

        optimizer.zero_grad()

        preds = bias_model(
            users,
            books
        )

        loss = criterion(
            preds,
            ratings
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Avg Loss: {avg_loss:.4f}"
    )

Epoch 1/10 | Avg Loss: 5.1585
Epoch 2/10 | Avg Loss: 5.0777
Epoch 3/10 | Avg Loss: 4.9974
Epoch 4/10 | Avg Loss: 4.9190
Epoch 5/10 | Avg Loss: 4.8456
Epoch 6/10 | Avg Loss: 4.7715
Epoch 7/10 | Avg Loss: 4.7004
Epoch 8/10 | Avg Loss: 4.6301
Epoch 9/10 | Avg Loss: 4.5633
Epoch 10/10 | Avg Loss: 4.4983


In [218]:
bias_model.eval()

with torch.no_grad():

    preds = bias_model(
        test_users.to(device, non_blocking=True),
        test_books.to(device, non_blocking=True)
    )

    rmse = torch.sqrt(
        ((preds - test_ratings.to(device, non_blocking=True)) ** 2).mean()
    )

print("RMSE:", rmse.item())

RMSE: 2.241673469543457


In [219]:
popular_books = (
    train_df
    .groupby("book_idx")
    .size()
    .sort_values(ascending=False)
)

popular_books.head(10)

book_idx
438    311
11     256
71     178
464    172
751    167
490    164
215    150
366    148
283    147
376    140
dtype: int64

In [220]:
idx2book = {
    idx: isbn
    for isbn, idx in book2idx.items()
}

In [221]:
import pandas as pd

books_df = pd.read_csv(
    "../data/processed/books_master.csv"
)

print(books_df.shape)
books_df.head()

C:\Users\anshu\AppData\Local\Temp\ipykernel_45500\4007194975.py:3: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  books_df = pd.read_csv(


(271360, 13)


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L,description,categories,language,page_count,thumbnail
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,NaN,NaN,NaN,NaN,NaN
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,NaN,NaN,NaN,NaN,NaN
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,NaN,NaN,NaN,NaN,NaN
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,NaN,NaN,NaN,NaN,NaN
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,NaN,NaN,NaN,NaN,NaN


In [222]:
books_df.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L', 'description',
       'categories', 'language', 'page_count', 'thumbnail'],
      dtype='str')

In [223]:
idx2book = {
    idx: isbn
    for isbn, idx in book2idx.items()
}

top_books = []

for idx in popular_books.head(10).index:

    isbn = idx2book[idx]

    title = books_df.loc[
        books_df["ISBN"] == isbn,
        "Book-Title"
    ].iloc[0]

    author = books_df.loc[
        books_df["ISBN"] == isbn,
        "Book-Author"
    ].iloc[0]

    top_books.append(
        (title, author)
    )

top_books

[('The Lovely Bones: A Novel', 'Alice Sebold'),
 ('The Da Vinci Code', 'Dan Brown'),
 ('The Red Tent (Bestselling Backlist)', 'Anita Diamant'),
 ("Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))",
  'J. K. Rowling'),
 ('Wild Animus', 'Rich Shapero'),
 ('The Secret Life of Bees', 'Sue Monk Kidd'),
 ("Where the Heart Is (Oprah's Book Club (Paperback))", 'Billie Letts'),
 ('Harry Potter and the Order of the Phoenix (Book 5)', 'J. K. Rowling'),
 ('Interview with the Vampire', 'Anne Rice'),
 ('Angels &amp; Demons', 'Dan Brown')]

In [224]:
top10_popular = (
    popular_books
    .head(10)
    .index
    .tolist()
)

print(top10_popular)

[438, 11, 71, 464, 751, 490, 215, 366, 283, 376]


In [225]:
hits = 0

for _, row in test_df.iterrows():

    true_book = row["book_idx"]

    if true_book in top10_popular:

        hits += 1

hit10 = hits / len(test_df)

print("Hit@10:", hit10)

Hit@10: 0.020816524124576408


In [226]:
user_book_matrix = (
    train_df
    .pivot_table(
        index="user_idx",
        columns="book_idx",
        values="Rating",
        fill_value=0
    )
)

print(user_book_matrix.shape)

(6197, 9009)


In [227]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(
    user_book_matrix.T
)

print(item_similarity.shape)

(9009, 9009)


In [228]:
item_similarity.nbytes / (1024**2)

619.2175369262695

In [229]:
import pandas as pd

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_book_matrix.columns,
    columns=user_book_matrix.columns
)

item_similarity_df.shape

(9009, 9009)

In [230]:
sample_book = user_book_matrix.columns[0]

sample_book

np.int64(0)

In [231]:
item_similarity_df.loc[sample_book] \
                  .sort_values(ascending=False) \
                  .head(10)

book_idx
0       1.000000
5783    0.256731
2126    0.243364
1999    0.229005
2906    0.211504
1787    0.207928
2284    0.203124
2292    0.200447
2280    0.197737
2281    0.195148
Name: 0, dtype: float64

In [232]:
train_df.head()

,User-ID,ISBN,Rating,user_idx,book_idx
103,276822,0375821813,9,0,1
109,276822,0786817070,10,0,2
152,276847,3442446414,10,1,4
168,276847,3551551677,10,1,5
169,276847,3551551685,10,1,6


In [233]:
%whos DataFrame


Variable             Type         Data/Info
-------------------------------------------
book_lookup          DataFrame    Shape: (9009, 2)
books_df             DataFrame    Shape: (271360, 13)
df                   DataFrame    Shape: (217729, 3)
explicit_ratings     DataFrame    Shape: (383842, 3)
group                DataFrame    Shape: (13, 5)
item_similarity_df   DataFrame    Shape: (9009, 9009)
ratings_cf           DataFrame    Shape: (261899, 5)
ratings_dense        DataFrame    Shape: (108344, 5)
ratings_master       DataFrame    Shape: (1031136, 3)
rec_df               DataFrame    Shape: (10, 5)
results_df           DataFrame    Shape: (5, 3)
subset               DataFrame    Shape: (217729, 3)
test_df              DataFrame    Shape: (6197, 5)
test_sample          DataFrame    Shape: (1, 5)
train_df             DataFrame    Shape: (101908, 5)
train_sample         DataFrame    Shape: (12, 5)
user0                DataFrame    Shape: (2, 7)
user_book_matrix     DataFrame    Shape

In [234]:
books_df.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L', 'description',
       'categories', 'language', 'page_count', 'thumbnail'],
      dtype='str')

In [235]:
books_df.head(2)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L,description,categories,language,page_count,thumbnail
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,NaN,NaN,NaN,NaN,NaN
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,NaN,NaN,NaN,NaN,NaN


In [236]:
import pandas as pd

def recommend_books(
    user_id,
    train_df,
    item_similarity_df,
    top_n=10
):

    user_ratings = train_df[
        train_df["user_idx"] == user_id
    ]

    scores = {}

    for _, row in user_ratings.iterrows():

        book = row["book_idx"]
        rating = row["Rating"]

        similar_books = item_similarity_df[book]

        for sim_book, sim_score in similar_books.items():

            if sim_book == book:
                continue

            scores[sim_book] = (
                scores.get(sim_book, 0)
                + sim_score * rating
            )

    already_read = set(
        user_ratings["book_idx"]
    )

    recommendations = (
        pd.Series(scores)
        .drop(labels=already_read, errors="ignore")
        .sort_values(ascending=False)
        .head(top_n)
    )

    return recommendations

In [237]:
sample_user = 0

recs = recommend_books(
    sample_user,
    train_df,
    item_similarity_df,
    top_n=10
)

recs

4458    2.415546
8463    2.279259
8616    2.222449
3899    2.040078
5253    1.939707
6412    1.935050
8625    1.908318
5349    1.876929
1087    1.871589
5175    1.839932
dtype: float64

In [238]:
books_df.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L', 'description',
       'categories', 'language', 'page_count', 'thumbnail'],
      dtype='str')

In [239]:
recs

4458    2.415546
8463    2.279259
8616    2.222449
3899    2.040078
5253    1.939707
6412    1.935050
8625    1.908318
5349    1.876929
1087    1.871589
5175    1.839932
dtype: float64

In [240]:
ratings_dense.head()

,User-ID,ISBN,Rating,user_idx,book_idx
99,276822,0060096195,10,0,0
103,276822,0375821813,9,0,1
109,276822,0786817070,10,0,2
137,276847,3404148576,8,1,3
152,276847,3442446414,10,1,4


In [241]:
ratings_dense.columns

Index(['User-ID', 'ISBN', 'Rating', 'user_idx', 'book_idx'], dtype='str')

In [242]:
train_df[
    train_df["user_idx"] == 0
]


,User-ID,ISBN,Rating,user_idx,book_idx
103,276822,0375821813,9,0,1
109,276822,0786817070,10,0,2


In [243]:
book_lookup = (
    ratings_dense[["book_idx", "ISBN"]]
    .drop_duplicates()
)

book_lookup.head()

,book_idx,ISBN
99,0,0060096195
103,1,0375821813
109,2,0786817070
137,3,3404148576
152,4,3442446414


In [244]:
recs

4458    2.415546
8463    2.279259
8616    2.222449
3899    2.040078
5253    1.939707
6412    1.935050
8625    1.908318
5349    1.876929
1087    1.871589
5175    1.839932
dtype: float64

In [245]:
rec_df = pd.DataFrame({
    "book_idx": recs.index,
    "score": recs.values
})

rec_df = rec_df.merge(
    book_lookup,
    on="book_idx",
    how="left"
)

rec_df = rec_df.merge(
    books_df[[
        "ISBN",
        "Book-Title",
        "Book-Author"
    ]],
    on="ISBN",
    how="left"
)

rec_df

,book_idx,score,ISBN,Book-Title,Book-Author
0,4458,2.415546,0786808551,"The Arctic Incident (Artemis Fowl, Book 2)",Eoin Colfer
1,8463,2.279259,0805073337,What Was She Thinking?: Notes on a Scandal: A ...,Zoe Heller
2,8616,2.222449,0786851473,Artemis Fowl: The Arctic Incident - Book #2 (A...,Eoin Colfer
3,3899,2.040078,0590407600,Magic School Bus: Inside the Earth (Magic Scho...,Joanna Cole
4,5253,1.939707,0064408639,The Austere Academy (A Series of Unfortunate E...,Lemony Snicket
5,6412,1.935050,0590484125,The Magic School Bus in the Haunted Museum: A ...,Joanna Cole
6,8625,1.908318,0345382293,If I'd Killed Him When I Met Him...: An Elizab...,Sharyn McCrumb
7,5349,1.876929,0140185003,The Quiet American (Penguin Twentieth Century ...,Graham Greene
8,1087,1.871589,067975833X,Confederates in the Attic : Dispatches from th...,TONY HORWITZ
9,5175,1.839932,0375709150,Driving over Lemons: An Optimist in Spain (Vin...,Chris Stewart


In [246]:
user0 = train_df[
    train_df["user_idx"] == 0
]

user0 = user0.merge(
    books_df[[
        "ISBN",
        "Book-Title",
        "Book-Author"
    ]],
    on="ISBN",
    how="left"
)

user0[[
    "Book-Title",
    "Book-Author",
    "Rating"
]]

,Book-Title,Book-Author,Rating
0,Hoot (Newbery Honor Book),CARL HIAASEN,9
1,"Artemis Fowl (Artemis Fowl, Book 1)",Eoin Colfer,10


In [247]:
len(test_df)

6197

In [248]:
def recommend_top_n(
    user_id,
    train_df,
    item_similarity_df,
    top_n=10
):

    user_ratings = train_df[
        train_df["user_idx"] == user_id
    ]

    scores = {}

    for _, row in user_ratings.iterrows():

        book = row["book_idx"]
        rating = row["Rating"]

        similar_books = item_similarity_df[book]

        for sim_book, sim_score in similar_books.items():

            if sim_book == book:
                continue

            scores[sim_book] = (
                scores.get(sim_book, 0)
                + sim_score * rating
            )

    already_read = set(
        user_ratings["book_idx"]
    )

    recommendations = (
        pd.Series(scores)
        .drop(labels=already_read, errors="ignore")
        .sort_values(ascending=False)
        .head(top_n)
        .index
        .tolist()
    )

    return recommendations

In [249]:
recommend_top_n(
    0,
    train_df,
    item_similarity_df,
    top_n=10
)


[4458, 8463, 8616, 3899, 5253, 6412, 8625, 5349, 1087, 5175]

In [250]:
coverage = (
    test_df["book_idx"]
    .nunique()
)

print("Unique test books:", coverage)

Unique test books: 3752


In [251]:
len(set(test_df["book_idx"]))

3752

In [252]:
import pandas as pd

def recommend_top_n_k(
    user_id,
    train_df,
    item_similarity_df,
    top_n=10,
    k=50
):

    user_ratings = train_df[
        train_df["user_idx"] == user_id
    ]

    scores = {}

    for _, row in user_ratings.iterrows():

        book = row["book_idx"]
        rating = row["Rating"]

        # Get only top-k similar books
        neighbors = (
            item_similarity_df[book]
            .sort_values(ascending=False)
            .iloc[1:k+1]      # skip self-similarity
        )

        for sim_book, sim_score in neighbors.items():

            scores[sim_book] = (
                scores.get(sim_book, 0)
                + sim_score * rating
            )

    already_read = set(
        user_ratings["book_idx"]
    )

    recommendations = (
        pd.Series(scores)
        .drop(labels=already_read, errors="ignore")
        .sort_values(ascending=False)
        .head(top_n)
        .index
        .tolist()
    )

    return recommendations

In [253]:
k_values = [10, 25, 50, 100, 200]

results = []

for k in k_values:

    hits = 0

    for _, row in test_df.iterrows():

        user_id = row["user_idx"]
        true_book = row["book_idx"]

        recommendations = recommend_top_n_k(
            user_id=user_id,
            train_df=train_df,
            item_similarity_df=item_similarity_df,
            top_n=10,
            k=k
        )

        if true_book in recommendations:
            hits += 1

    hit10 = hits / len(test_df)

    results.append({
        "K": k,
        "Hits": hits,
        "Hit@10": hit10
    })

    print(f"K={k} | Hits={hits} | Hit@10={hit10:.4f}")

K=10 | Hits=368 | Hit@10=0.0594
K=25 | Hits=346 | Hit@10=0.0558
K=50 | Hits=330 | Hit@10=0.0533
K=100 | Hits=312 | Hit@10=0.0503
K=200 | Hits=301 | Hit@10=0.0486


In [254]:
results_df = pd.DataFrame(results)

results_df

,K,Hits,Hit@10
0,10,368,0.059384
1,25,346,0.055833
2,50,330,0.053252
3,100,312,0.050347
4,200,301,0.048572


In [255]:
print(train_df.shape)
print(test_df.shape)

print(train_df.columns)

(101908, 5)
(6197, 5)
Index(['User-ID', 'ISBN', 'Rating', 'user_idx', 'book_idx'], dtype='str')


In [256]:
n_users = train_df["user_idx"].nunique()
n_books = train_df["book_idx"].nunique()

print("Users:", n_users)
print("Books:", n_books)

Users: 6197
Books: 9009


In [257]:
[x for x in globals().keys() if "similarity" in x.lower()]

['cosine_similarity', 'item_similarity', 'item_similarity_df']

In [258]:
[x for x in globals().keys() if "item" in x.lower()]

['item_similarity', 'item_similarity_df']

In [259]:
print("user_book_matrix" in globals())
print("item_similarity_df" in globals())

True
True


In [260]:
import pickle

with open(
    "../models/item_similarity_df.pkl",
    "wb"
) as f:
    pickle.dump(
        item_similarity_df,
        f
    )

print("Saved!")

Saved!


In [261]:
train_df.to_csv(
    "../data/processed/train_df.csv",
    index=False
)

test_df.to_csv(
    "../data/processed/test_df.csv",
    index=False
)

In [262]:
with open(
    "../models/user2idx.pkl",
    "wb"
) as f:
    pickle.dump(user2idx, f)

with open(
    "../models/book2idx.pkl",
    "wb"
) as f:
    pickle.dump(book2idx, f)

In [263]:
idx2user = {
    idx: user
    for user, idx in user2idx.items()
}

idx2book = {
    idx: isbn
    for isbn, idx in book2idx.items()
}

In [264]:
with open(
    "../models/idx2book.pkl",
    "wb"
) as f:
    pickle.dump(idx2book, f)

with open(
    "../models/idx2user.pkl",
    "wb"
) as f:
    pickle.dump(idx2user, f)